In [ ]:
import ROOT, math, numpy as np
import random

In [ ]:
File = ROOT.TFile("LEP1MC1994_recons_aftercut-001.root")
#no detector simulation for matchbox
File.ls()

In [ ]:
Tree = File.Get("t")
Tree.Print()

In [ ]:
Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
Tree.Draw("nref")
Canvas.Draw()

In [ ]:
#Comparison of total energy to Distance
def jetshape(inputfilename, outputfilename, JetTreename, particletreename):
    File = ROOT.TFile("LEP1Data1994P1_recons_aftercut-MERGED.root")
    outFile = ROOT.TFile(outputfilename, "recreate")
    Hist12 = ROOT.TH1D("HIST12", "Total Energy vs. Distance", 80, 0, 2)
    Counter = 0
    TreeGen = File.Get(JetTreename)
    TreeParticle = File.Get(particletreename)
    TotalNumberOfCollisions = TreeParticle.GetEntries("nParticle")
    TargetNumberOfCollisions = TotalNumberOfCollisions * 0.01
    CurrentlyProcessed = 0
    #processing event collisions
    for EventGen, ParticleGen in zip(TreeGen, TreeParticle):
         CurrentlyProcessed = CurrentlyProcessed + 1
         if CurrentlyProcessed > TargetNumberOfCollisions:
             break
         GenPX = [EventGen.jtpt[i] * math.cos(EventGen.jteta[i]) for i in range(EventGen.nref)]
         GenPY = [EventGen.jtpt[i] * math.sin(EventGen.jteta[i]) for i in range(EventGen.nref)]
         GenPZ = [EventGen.jtpt[i] * math.sinh(EventGen.jteta[i]) for i in range(EventGen.nref)]
         GenP =  [EventGen.jtpt[i] * math.cosh(EventGen.jteta[i]) for i in range(EventGen.nref)]
         GenE = [math.sqrt(EventGen.jtm[i]**2 + GenP[i]**2) for i in range(EventGen.nref)]
        #E1
         PGPX = [ParticleGen.pt[i] * math.cos(ParticleGen.eta[i]) for i in range(ParticleGen.nParticle)]
         PGPY = [ParticleGen.pt[i] * math.sin(ParticleGen.eta[i]) for i in range(ParticleGen.nParticle)]
         #print([ParticleGen.eta[i] for i in range(ParticleGen.nParticle)])
         PGPZ = [ParticleGen.pt[i] * math.sinh(100 if ParticleGen.eta[i] > 100 else (-100 if ParticleGen.eta[i] < -100 else ParticleGen.eta[i])) for i in range(ParticleGen.nParticle)]
         PGP = [ParticleGen.pt[i] * math.cosh(100 if ParticleGen.eta[i] > 100 else (-100 if ParticleGen.eta[i] < -100 else ParticleGen.eta[i])) for i in range(ParticleGen.nParticle)]
         PGE = [math.sqrt(ParticleGen.mass[i]**2 + PGP[i]**2) for i in range(ParticleGen.nParticle)]
         #E1prime
         for iG in range(len(GenPX)):
             #miscellaneous below
             # we want to do both <40 and >40 want as well 10-20; 20-30; 30-40; 40+ (R8; MCt v. MCtgen) (MCt and Datat)
             #TotalEnergy = TreeParticle.GetEntries("Energy")
             #TargetEnergy = TotalEnergy * 0.1
             #ProcessedEnergy = 0
             #ProcessedEnergy = ProcessedEnergy + 1
             #if ProcessedEnergy > TargetEnergy:
                 #break
             if GenE[iG] > 40: 
                 continue
             Counter = Counter + 1
             for iP in range(ParticleGen.nParticle):
                 if PGP[iP] == 0 or GenP[iG] == 0:
                     continue
                 Cosine = (((GenPX[iG] * PGPX[iP]) + (GenPY[iG] * PGPY[iP]) + (GenPZ[iG] * PGPZ[iP])) / (GenP[iG] * PGP[iP]))
                 if Cosine > 1: Cosine = 1
                 if Cosine < -1: Cosine = -1
                 Distance = np.arccos(Cosine)
                 if Distance < 0.8:
                     Hist12.GetXaxis().SetRangeUser(0, 0.8)
                     Hist12.Fill(Distance +random.gauss(0, 0.005), PGE[iP])
    Hist12.Scale(1 / Counter)
    for i in range (0, 80):
        r = Hist12.GetXaxis(). GetBinLowEdge(i + 1)
        R = Hist12.GetXaxis().GetBinUpEdge(i + 1)
        x = Hist12.GetBinContent(i + 1)
        Area = math.pi*(R*R - r*r)
        avg = x / Area
        Hist12.SetBinContent(i + 1, avg)
    
    Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
    #Hist2.GetXaxis().SetRangeUser(0, 0.8)
    #Hist3.GetXaxis().SetRangeUser(0, 0.8)
    Hist12.Draw()
    #Canvas.SetLogy()
    #delete previous line when doing new type of running
    outFile.cd()
    Hist12.Write()
    outFile.Close()

In [ ]:
jetshape("LEP1Data1994P1_recons_aftercut-MERGED.root", "LEPD>40t.root", "akR8ESchemeJetTree", "t")

In [ ]:
#Different simulations contrasted against the above code 
Canvas = ROOT.TCanvas("Canvas", "", 1024, 768)
Canvas.SetLogy()
File1 = ROOT.TFile("MC>40t.root")
#different simulation matchbox vs. sherpa vs. LEPIMC
#with and without detector effects which is LEPIMC with t and tgen two histograms
#everything under LEPIMC ALEPHMC
# with detector vs. data LEPIMC vs. LEPIData 
File2 = ROOT.TFile("LEPD>40t.root")
#File3 = ROOT.TFile("reconLEPMCtR8.root")

Hist1 = File1.Get("HIST12")
Hist2 = File2.Get("HIST12")
#Hist3 = File3.Get("HIST12")

Hist1.SetLineColor(ROOT.kBlue)
Hist2.SetLineColor(ROOT.kRed)
#Hist3.SetLineColor(ROOT.kGreen)

Legend = ROOT.TLegend(0.5, 0.8, 0.8, 0.6)
Legend.AddEntry(Hist1, "LEPMCtR8>40", "lp")
Legend.AddEntry(Hist2, "LEPDtR8>40", "lp")
#Legend.AddEntry(Hist3, "LEPMCR8", "lp")


Hist1.SetMarkerStyle(20)
Hist1.SetMarkerColor(ROOT.kBlue)
Hist2.SetMarkerStyle(20)
Hist2.SetMarkerColor(ROOT.kRed)
#Hist3.SetMarkerStyle(20)
#Hist3.SetMarkerColor(ROOT.kGreen)
Hist1.Draw();
Hist2.Draw("same")
#Hist3.Draw("same")

#Do everything with R8 and <0.8 distance 
#
Canvas.Draw()
Legend.Draw()